In [9]:
import numpy as np
from helpers import *
from implementations import * 


DATA_PATH = "./dataset/"

### Loading data

In [10]:
# Load data from CSV files
print("Loading data...")
x_train, x_test, y_train, train_ids, test_ids = load_csv_data(DATA_PATH)
print("Data loaded")

print("Negative class rate:", np.mean(y_train == -1))
print("Positive class rate:", np.mean(y_train == 1))

Loading data...
Data loaded
Negative class rate: 0.911697929205967
Positive class rate: 0.08830207079403295


There is a strong imbalance in the dataset. Only about 8.8 % of examples are positive.

A model that predicts every example as negative might get very high accuracy, but the positive-class F1 score would be very low. 

### Preprocessing of data

Before applying a model to our data, it first needs to be preprocessed. 

We start by performing one-hot encoding to address the problem of categorical features. A linear model treats categories  (e.g. $1, 2, 3, 4, 5$) as numbers, implying that category $5$ is "five times greater" than category $1$. One-hot encoding turns them into independent binary values, allowing  logistic regression weights ($w$) to learn a distinct coefficient for each specific category independently.

We then search for columns in the training set with a percentage of Nan values higher than 90%, because we assume that a too high percentage of missing values makes them irrelevant. These columns are removed in both the training and test data sets, because the model must be trained and evaluated using the same features.

In the remaining columns, Nan values are replaced by column medians. Column medians are chosen over column means, because medians are typically less sensitive to outliers than means. The medians are calculated using the training set only, then those same medians are used to fill missing values in both the training and the test set. This prevents data leakage from the test set into the training process.

The standardization parameters (mean, standard deviation) are calculated based on the training set. Then the features of both the training and the test sets are Z-normalized by removing the mean and dividing by the standard deviation. Again, only training statistics are used for both datasets to avoid influence of the test set on prepropressing and assuring consistency.

Finally, a column of 1s is added at the beginning of the training and test sets to account for the bias term (w0, intercept) in our model.

In [11]:
def preprocess_data(x_train, x_test, nan_threshold=0.90):
    """
    Perform preprocessing on training and testing datasets. 
    Includes:
     1. Removing columns with excessive Nan values.
     2. One-hot encoding of categorical features
     3. Replacing missing values with column medians (median imputation).
     4. Z-score standardization.
     5. Adding bias column of ones.
    
    Parameters:
    - x_train: array of raw training features 
    - x_test: array of raw testing features 
    - nan_threshold: column NaN percentage limit to keep (e.g. 0.90 = keep if <= 90% NaN)
    
    Returns:
    - x_train_final: filtered, imputed and scaled training set 
    - x_test_final: filtered, imputed and scaled test set 
    """
    print("--- Starting Preprocessing ---")
    
    # 1. Remove columns with high NaN ratios (> nan_threshold)
    nan_ratios_columns = np.isnan(x_train).mean(axis=0)
    columns_to_keep = nan_ratios_columns <= nan_threshold
    
    x_tr_filtered = x_train[:, columns_to_keep].copy()
    x_te_filtered = x_test[:, columns_to_keep].copy()

    nb_columns_removed = np.sum(~columns_to_keep) # The ~ operator performs a logical NOT on Boolean values --> False becomes True and vice vera
    percentage_nan_threshold = nan_threshold * 100

    print(f'Number of removed columns with >{percentage_nan_threshold:.0f}% NaNs: {nb_columns_removed}')
    print(f'Remaining features: {x_tr_filtered.shape[1]}')

    # 2. One-hot encode categorical features first (or right after filtering)
    x_tr_encoded, x_te_encoded = one_hot_encode_categorical(x_tr_filtered, x_te_filtered)
    print(f'Features shape after one-hot encoding: {x_tr_encoded.shape[1]}')
    
    # 3. Replace NaN values by column median values (operating on the encoded arrays)
    train_medians = np.nanmedian(x_tr_encoded, axis=0) 
    train_medians = np.nan_to_num(train_medians, nan=0.0) 
    
    for col_idx in range(x_tr_filtered.shape[1]):
        # Replaces missing values in training set with corresponding column median
        train_nan = np.isnan(x_tr_encoded[:, col_idx])
        x_tr_filtered[train_nan, col_idx] = train_medians[col_idx]
        
        # Replaces missing values in testing set with corresponding column median from training set 
        test_nan = np.isnan(x_te_encoded[:, col_idx])
        x_te_filtered[test_nan, col_idx] = train_medians[col_idx]
        
    print("Missing values replaced using medians from the training set.")
    
    # 3. Standardization (Z-score normalization)
    mean = np.mean(x_tr_encoded, axis=0)
    std = np.std(x_tr_encoded, axis=0)
    
    # Special case: division by zero --> std replaced by 1.0 for constant columns
    std[std == 0] = 1.0
    
    x_tr_normalized = (x_tr_encoded - mean) / std
    x_te_normalized = (x_te_encoded - mean) / std
    print("Features standardized (zero mean, unit variance).")
    
    # 4. Add a column of 1s at the beginning for the bias weight (w_0)
    bias_train = np.ones((x_tr_normalized.shape[0], 1))
    bias_test = np.ones((x_te_normalized.shape[0], 1))
    
    x_tr_final = np.hstack((bias_train, x_tr_normalized))
    x_te_final = np.hstack((bias_test, x_te_normalized))
    
    print(f"Final processed training shape: {x_tr_final.shape}")
    print(f"Final processed testing shape: {x_te_final.shape}")
    print("--- Preprocessing Complete ---\n")
    
    return x_tr_final, x_te_final

What we could do :
- Handle categorical variables : Categorical features might be mistakenly treated as continuous. Therefore, categorical columns should be converted into binary indicator variables if applicable.

### 1. Experiment - Compare performance and choose threshold

In [12]:
def weighted_reg_logistic_regression(
    y, tx, lambda_, initial_w, max_iters, gamma
):
    """Class-weighted regularized logistic regression."""
    w = initial_w.copy()

    # Calculate weights from the training labels
    negative_rate = np.mean(y.ravel() == 0)
    positive_rate = np.mean(y.ravel() == 1)

    weight_negative = 1.0 / (2 * negative_rate)
    weight_positive = 1.0 / (2 * positive_rate)

    # Give each training example its class weight
    sample_weights = np.where(
        y == 1, weight_positive, weight_negative
    ).reshape(-1, 1)

    for _ in range(max_iters):
        probabilities = sigmoid(tx @ w)

        # Weighted logistic gradient
        gradient = (
            tx.T @ (sample_weights * (probabilities - y))
            / len(y)
        )

        # Regularization, matching your original function
        gradient += 2 * lambda_ * w

        w = w - gamma * gradient

    # Calculate weighted logistic loss safely
    probabilities = np.clip(sigmoid(tx @ w), 1e-15, 1 - 1e-15)

    loss = -np.mean(
        sample_weights * (
            y * np.log(probabilities)
            + (1 - y) * np.log(1 - probabilities)
        )
    ) + lambda_ * np.sum(w**2)

    return w, loss


### 1.1. Experiment A : Normal logistic regression

In [13]:
# 1. Split the original data BEFORE preprocessing
rng = np.random.default_rng(42)
indices = rng.permutation(len(y_train))

# 80% train, 20% validation
split = int(0.8 * len(indices))
train_idx = indices[:split]
val_idx = indices[split:]

x_tr_raw = x_train[train_idx] 
x_val_raw = x_train[val_idx] 

y_tr = y_train[train_idx]  
y_val = y_train[val_idx]

# 2. Fit preprocessing on training data only, with Nan threshold of 90%
x_tr, x_val = preprocess_data(
    x_tr_raw, x_val_raw, nan_threshold=0.90
)

# Convert labels from {-1, +1} to {0, 1}
y_tr_bin = np.where(y_tr == -1, 0, 1).reshape(-1, 1)
y_val_bin = (y_val == 1).astype(int).ravel()


# 3. Calculate class weights
negative_rate = np.mean(y_tr_bin.ravel() == 0)
positive_rate = np.mean(y_tr_bin.ravel() == 1)

weight_negative = 1.0 / (2 * negative_rate)
weight_positive = 1.0 / (2 * positive_rate)

print("Training negative rate:", negative_rate)
print("Training positive rate:", positive_rate)
print("Negative class weight:", weight_negative)
print("Positive class weight:", weight_positive)


# 4. Train the same regularized logistic regression
w_init = np.zeros((x_tr.shape[1], 1))

# Using non-weighted regularized logistic regression
w_opt, loss = reg_logistic_regression(
    y_tr_bin,
    x_tr,
    0.0005,       # regularization
    w_init,
    300,          # iterations
    0.5           # learning rate
)

# 5. Calculate validation probabilities
probs = np.asarray(sigmoid(x_val @ w_opt)).ravel()

# 6. Evaluate multiple decision thresholds
thresholds = np.arange(0.10, 0.801, 0.01)
results = []

for threshold in thresholds:
    pred = (probs >= threshold).astype(int)
    
    tp = np.sum((pred == 1) & (y_val_bin == 1)) # tp : true positives
    fp = np.sum((pred == 1) & (y_val_bin == 0)) # fp : false positives
    fn = np.sum((pred == 0) & (y_val_bin == 1)) # fn : false negatives

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    accuracy = np.mean(pred == y_val_bin)
    results.append((f1, threshold, precision, recall, accuracy))

# Show the best thresholds by validation F1
results.sort(reverse=True)

print("\nBest validation thresholds:")
print("F1       Threshold  Precision  Recall  Accuracy")

for f1, threshold, precision, recall, accuracy in results[:10]:
    print(
        f"{f1:.4f}   {threshold:.2f}       "
        f"{precision:.4f}     {recall:.4f}  {accuracy:.4f}"
    )


--- Starting Preprocessing ---
Number of removed columns with >90% NaNs: 99
Remaining features: 222


KeyboardInterrupt: 

### 1.2. Experiment B : Weighted logitisc regression

In [ ]:
# Experiment B: Normal logistic regression


# 1. Split the original data BEFORE preprocessing
rng = np.random.default_rng(42)
indices = rng.permutation(len(y_train))

# 80% train, 20% validation
split = int(0.8 * len(indices))
train_idx = indices[:split]
val_idx = indices[split:]

x_tr_raw = x_train[train_idx] 
x_val_raw = x_train[val_idx] 

y_tr = y_train[train_idx]  
y_val = y_train[val_idx]

# 2. Fit preprocessing on training data only, with Nan threshold of 90%
x_tr, x_val = preprocess_data(
    x_tr_raw, x_val_raw, nan_threshold=0.90
)

# Convert labels from {-1, +1} to {0, 1}
y_tr_bin = np.where(y_tr == -1, 0, 1).reshape(-1, 1)
y_val_bin = (y_val == 1).astype(int).ravel()


# 3. Calculate class weights
negative_rate = np.mean(y_tr_bin.ravel() == 0)
positive_rate = np.mean(y_tr_bin.ravel() == 1)

weight_negative = 1.0 / (2 * negative_rate)
weight_positive = 1.0 / (2 * positive_rate)

print("Training negative rate:", negative_rate)
print("Training positive rate:", positive_rate)
print("Negative class weight:", weight_negative)
print("Positive class weight:", weight_positive)


# 4. Train the same regularized logistic regression
w_init = np.zeros((x_tr.shape[1], 1))

# Using weighted regularized logistic regression
w_opt, loss = weighted_reg_logistic_regression(
    y_tr_bin,
    x_tr,
    0.0005,       # regularization
    w_init,
    300,          # iterations
    0.5           # learning rate
)

# 5. Calculate validation probabilities
probs = np.asarray(sigmoid(x_val @ w_opt)).ravel()

# 6. Evaluate multiple decision thresholds
thresholds = np.arange(0.10, 0.801, 0.01)
results = []

for threshold in thresholds:
    pred = (probs >= threshold).astype(int)
    
    tp = np.sum((pred == 1) & (y_val_bin == 1)) # tp : true positives
    fp = np.sum((pred == 1) & (y_val_bin == 0)) # fp : false positives
    fn = np.sum((pred == 0) & (y_val_bin == 1)) # fn : false negatives

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    accuracy = np.mean(pred == y_val_bin)
    results.append((f1, threshold, precision, recall, accuracy))

# Show the best thresholds by validation F1
results.sort(reverse=True)

print("\nBest validation thresholds:")
print("F1       Threshold  Precision  Recall  Accuracy")

for f1, threshold, precision, recall, accuracy in results[:10]:
    print(
        f"{f1:.4f}   {threshold:.2f}       "
        f"{precision:.4f}     {recall:.4f}  {accuracy:.4f}"
    )


--- Starting Preprocessing ---
Number of removed columns with >90% NaNs: 99
Remaining features: 222
Auto-detected or selected categorical columns at indices: [1, 3, 5, 6, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 47, 48, 49, 50, 51, 52, 53, 54, 56, 57, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 83, 91, 92, 93, 94, 95, 96, 98, 99, 100, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 124, 125, 126, 128, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 150, 155, 156, 157, 158, 159, 160, 161, 162, 164, 166, 173, 174, 175, 176, 179, 180, 181, 182, 183, 184, 185, 190, 191, 199, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221]
Features shape after one-hot encoding: 868
Missing values replaced using medians from the training set.
Features standardized (zero mean, unit variance).
Final 

### 2. Mini-batch SGD and stratified-fold helpers

In [ ]:

def make_stratified_folds(y, n_folds=5, seed=42):
    """Create stratified folds for binary labels encoded as 0 and 1."""
    y = np.asarray(y).ravel()
    rng = np.random.default_rng(seed)

    class_indices = []
    for label in (0, 1):
        indices = np.flatnonzero(y == label)
        rng.shuffle(indices)
        class_indices.append(np.array_split(indices, n_folds))

    folds = []
    for k in range(n_folds):
        fold_indices = np.concatenate([
            class_indices[0][k],
            class_indices[1][k]
        ])
        rng.shuffle(fold_indices)
        folds.append(fold_indices)

    return folds


def sigmoid_safe(z):
    """Numerically stable sigmoid."""
    z = np.asarray(z)
    result = np.empty_like(z, dtype=float)

    positive = z >= 0
    result[positive] = 1.0 / (1.0 + np.exp(-z[positive]))

    exp_z = np.exp(z[~positive])
    result[~positive] = exp_z / (1.0 + exp_z)

    return result


def train_logistic_sgd(
    y, X, lambda_=0.0005, learning_rate=0.05,
    epochs=12, batch_size=2048, weighted=False, seed=42
):
    """
    Train binary logistic regression using shuffled mini-batch SGD.

    y must contain 0/1 labels.
    X is expected to include the bias column from preprocess_data.
    """
    y = np.asarray(y, dtype=float).reshape(-1, 1)
    X = np.asarray(X, dtype=float)

    n_samples, n_features = X.shape
    w = np.zeros((n_features, 1))

    if weighted:
        negative_rate = np.mean(y.ravel() == 0)
        positive_rate = np.mean(y.ravel() == 1)

        class_weights = np.where(
            y == 1,
            1.0 / (2.0 * positive_rate),
            1.0 / (2.0 * negative_rate)
        )
    else:
        class_weights = np.ones_like(y)

    # Avoid regularizing the final bias column.
    regularization_mask = np.ones((n_features, 1))
    regularization_mask[-1, 0] = 0.0

    rng = np.random.default_rng(seed)

    for epoch in range(epochs):
        shuffled_indices = rng.permutation(n_samples)
        current_lr = learning_rate / (1.0 + 0.1 * epoch)

        for start in range(0, n_samples, batch_size):
            batch_idx = shuffled_indices[start:start + batch_size]

            X_batch = X[batch_idx]
            y_batch = y[batch_idx]
            weights_batch = class_weights[batch_idx]

            probabilities = sigmoid_safe(X_batch @ w)

            gradient = (
                X_batch.T @ (
                    weights_batch * (probabilities - y_batch)
                ) / len(batch_idx)
            )

            gradient += (
                2.0 * lambda_ * regularization_mask * w
            )

            w -= current_lr * gradient

    return w

### 3. 5-fold cross-validation

In [ ]:
# Convert original labels {-1, +1} to binary labels {0, 1}.
y_all_binary = np.where(
    np.asarray(y_train).ravel() == -1, 0, 1
)

X_raw = np.asarray(x_train)
n_folds = 5

folds = make_stratified_folds(
    y_all_binary, n_folds=n_folds, seed=42
)

# Each row receives one prediction from a model that did not train on it.
oof_normal = np.zeros(len(y_all_binary))
oof_weighted = np.zeros(len(y_all_binary))

for fold_num, val_idx in enumerate(folds, start=1):
    print(f"\n========== Fold {fold_num}/{n_folds} ==========")

    train_idx = np.setdiff1d(
        np.arange(len(y_all_binary)),
        val_idx,
        assume_unique=True
    )

    X_fold_train_raw = X_raw[train_idx]
    X_fold_val_raw = X_raw[val_idx]

    y_fold_train = y_all_binary[train_idx]
    y_fold_val = y_all_binary[val_idx]

    # Fit preprocessing only on this fold's training data.
    X_fold_train, X_fold_val = preprocess_data(
        X_fold_train_raw,
        X_fold_val_raw,
        nan_threshold=0.90
    )

    print("Training normal SGD...")
    w_normal = train_logistic_sgd(
        y_fold_train,
        X_fold_train,
        lambda_=0.0005,
        learning_rate=0.05,
        epochs=12,
        batch_size=2048,
        weighted=False,
        seed=100 + fold_num
    )

    print("Training weighted SGD...")
    w_weighted = train_logistic_sgd(
        y_fold_train,
        X_fold_train,
        lambda_=0.0005,
        learning_rate=0.05,
        epochs=12,
        batch_size=2048,
        weighted=True,
        seed=100 + fold_num
    )

    # Save probabilities for the held-out fold.
    oof_normal[val_idx] = sigmoid_safe(
        X_fold_val @ w_normal
    ).ravel()

    oof_weighted[val_idx] = sigmoid_safe(
        X_fold_val @ w_weighted
    ).ravel()

    print(
        f"Fold {fold_num} complete. "
        f"Validation rows: {len(val_idx):,}"
    )

print("\nCross-validation predictions are ready.")


========== Fold 1/5 ==========
--- Starting Preprocessing ---
Number of removed columns with >90% NaNs: 99
Remaining features: 222
Auto-detected or selected categorical columns at indices: [1, 3, 5, 6, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 47, 48, 49, 50, 51, 52, 53, 54, 56, 57, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 83, 91, 92, 93, 94, 95, 96, 98, 99, 100, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 124, 125, 126, 128, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 150, 155, 156, 157, 158, 159, 160, 161, 162, 164, 166, 173, 174, 175, 176, 179, 180, 181, 182, 183, 184, 185, 190, 191, 199, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221]
Features shape after one-hot encoding: 868
Missing values replaced using medians from the training set.
Features standardized (z

KeyboardInterrupt: 

### 4. Compare F1, precision, and recall

In [ ]:
def calculate_metrics(y_true, probabilities, threshold=0.5):
    y_true = np.asarray(y_true).ravel()
    y_pred = (probabilities >= threshold).astype(int)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )
    accuracy = (tp + tn) / len(y_true)

    return {
        "F1": f1,
        "Precision": precision,
        "Recall": recall,
        "Accuracy": accuracy
    }


def find_best_threshold(y_true, probabilities):
    thresholds = np.arange(0.05, 0.801, 0.01)
    results = []

    for threshold in thresholds:
        metrics = calculate_metrics(
            y_true, probabilities, threshold
        )
        results.append({
            "Threshold": round(float(threshold), 2),
            **metrics
        })

    results.sort(key=lambda row: row["F1"], reverse=True)
    return results


normal_results = find_best_threshold(
    y_all_binary, oof_normal
)
weighted_results = find_best_threshold(
    y_all_binary, oof_weighted
)

print("NORMAL SGD — best thresholds")
print("F1       Threshold  Precision  Recall  Accuracy")
for row in normal_results[:10]:
    print(
        f'{row["F1"]:.4f}   '
        f'{row["Threshold"]:.2f}       '
        f'{row["Precision"]:.4f}     '
        f'{row["Recall"]:.4f}  '
        f'{row["Accuracy"]:.4f}'
    )

print("\nWEIGHTED SGD — best thresholds")
print("F1       Threshold  Precision  Recall  Accuracy")
for row in weighted_results[:10]:
    print(
        f'{row["F1"]:.4f}   '
        f'{row["Threshold"]:.2f}       '
        f'{row["Precision"]:.4f}     '
        f'{row["Recall"]:.4f}  '
        f'{row["Accuracy"]:.4f}'
    )

NORMAL SGD — best thresholds
F1       Threshold  Precision  Recall  Accuracy
0.4098   0.20       0.3247     0.5553  0.8588
0.4097   0.21       0.3340     0.5296  0.8652
0.4095   0.22       0.3437     0.5066  0.8710
0.4088   0.19       0.3150     0.5823  0.8513
0.4066   0.23       0.3521     0.4810  0.8760
0.4054   0.18       0.3044     0.6067  0.8428
0.4047   0.24       0.3620     0.4588  0.8808
0.4024   0.17       0.2948     0.6338  0.8338
0.4014   0.25       0.3713     0.4367  0.8850
0.3978   0.16       0.2846     0.6605  0.8234

WEIGHTED SGD — best thresholds
F1       Threshold  Precision  Recall  Accuracy
0.4098   0.72       0.3307     0.5385  0.8630
0.4093   0.71       0.3239     0.5556  0.8584
0.4092   0.73       0.3371     0.5206  0.8673
0.4086   0.70       0.3178     0.5720  0.8538
0.4076   0.74       0.3431     0.5020  0.8712
0.4076   0.75       0.3515     0.4849  0.8755
0.4071   0.69       0.3115     0.5875  0.8489
0.4055   0.68       0.3054     0.6031  0.8438
0.4051   0.76  

### 5. Select the winner and train on all labeled data

In [ ]:
# Select the model with the highest out-of-fold F1.
best_normal = normal_results[0]
best_weighted = weighted_results[0]

if best_weighted["F1"] > best_normal["F1"]:
    chosen_model = "weighted"
    best_result = best_weighted
else:
    chosen_model = "normal"
    best_result = best_normal

best_threshold = best_result["Threshold"]

print("Selected model:", chosen_model)
print("Selected OOF threshold:", best_threshold)
print("Best OOF F1:", round(best_result["F1"], 4))

# Fit preprocessing on all labeled training rows.
X_full, X_test_proc = preprocess_data(
    x_train, x_test, nan_threshold=0.90
)

# Retrain the selected model on all labeled examples.
w_final = train_logistic_sgd(
    y_all_binary,
    X_full,
    lambda_=0.0005,
    learning_rate=0.05,
    epochs=12,
    batch_size=2048,
    weighted=(chosen_model == "weighted"),
    seed=42
)

# Predict probabilities, then convert back to {-1, +1}.
test_probabilities = sigmoid_safe(
    X_test_proc @ w_final
).ravel()

test_predictions = np.where(
    test_probabilities >= best_threshold, 1, -1
)

print("Positive test predictions:", np.sum(test_predictions == 1))
print("Negative test predictions:", np.sum(test_predictions == -1))

create_csv_submission(
    test_ids,
    test_predictions,
    "sgd_logistic_submission.csv"
)

print("Submission file created: sgd_logistic_submission.csv")

Selected model: normal
Selected OOF threshold: 0.2
Best OOF F1: 0.4098
--- Starting Preprocessing ---
Number of removed columns with >90% NaNs: 99
Remaining features: 222
Missing values replaced using medians from the training set.
Features standardized (zero mean, unit variance).
Final processed training shape: (328135, 223)
Final processed testing shape: (109379, 223)
--- Preprocessing Complete ---

Positive test predictions: 16114
Negative test predictions: 93265
Submission file created: sgd_logistic_submission.csv


### Determination of model overfits or underfits

Underfitting
- Symptoms : 
    - High training loss / low training F1
    - High validation loss / low validation F1
- Causes : 
    - Model too simple
    - Regularization (lambda) too high
    - Not enough training iterations

Overfitting
- Symptoms : 
    - Very low training loss / high training F1
    - High validation loss / low validation F1
- Causes :
    - Model too complex
    - No regularization or too small
    - Too many noisy features

